In [1]:
# !pip install "google-cloud-bigquery-storage>=2.0.0"
!pip install --upgrade pip setuptools wheel
!pip install seqeval flair evaluate tqdm pytorch-crf fasttext-wheel huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 36.7 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (

In [2]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import random
import xml.etree.ElementTree as ET
from pathlib import Path
import torch
SEED = 23022006
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ENABLED_SOURCES = ['biomedbert', 'flair', 'fasttext']
# Đổi cờ rồi chạy lại từ đầu để tái tạo nguồn và optimizer.
TRAINABLE_SOURCES = {'biomedbert': False, 'flair': False, 'fasttext': False}
EMBEDDING_LRS = {'biomedbert': 1e-4, 'flair': 1e-5, 'fasttext': 1e-4}
MODEL_NAME = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
FLAIR_NAMES = ['pubmed-forward', 'pubmed-backward']
EPOCHS, PATIENCE, BATCH_SIZE, LR = 40, 5, 32, 1e-3
ANNOTATION_POLICY = 'project'  # 'strict': dừng khi annotation không khớp token BIO
RUN_NAME = '-'.join(ENABLED_SOURCES) + '-online-' + '-'.join(f'{k}{int(TRAINABLE_SOURCES[k])}' for k in ENABLED_SOURCES) + '-' + ANNOTATION_POLICY
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')) / 'multi_embedding_fusion' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Có thể trỏ tới model.bin đã gắn từ Kaggle Dataset.
FASTTEXT_PATH = OUTPUT_DIR.parent / 'pretrained' / 'fasttext-en-vectors' / 'model.bin'
DOWNLOAD_FASTTEXT = True
ENTITY_TYPES = ['drug', 'brand', 'group', 'drug_n']
LABELS = ['O'] + [f'{p}-{t}' for t in ENTITY_TYPES for p in ('B', 'I')]
label2id = {label: i for i, label in enumerate(LABELS)}
if not ENABLED_SOURCES or len(set(ENABLED_SOURCES)) != len(ENABLED_SOURCES) or set(ENABLED_SOURCES) - {'biomedbert', 'flair', 'fasttext'}:
    raise ValueError('ENABLED_SOURCES không hợp lệ')
print(DEVICE, ENABLED_SOURCES)


cuda ['biomedbert', 'flair', 'fasttext']


In [3]:
import json
import re
import hashlib
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torchcrf import CRF

TOKEN_PATTERN = r"\w+|[^\w\s]"


def tokenize_words(text):
    matches = list(re.finditer(TOKEN_PATTERN, text))
    return [m.group() for m in matches], [(m.start(), m.end()) for m in matches]



In [4]:
def prepare_example(ex, label2id, policy="strict"):
    if policy not in {"strict", "project"}:
        raise ValueError(f"Unknown annotation policy: {policy}")
    words, offsets = tokenize_words(ex['text'])
    labels = [label2id['O']] * len(words)
    owners, issues = {}, []
    if not words:
        issues.append(dict(reason='empty_sentence', fatal=True))
    seen = set()
    # Deterministic longest-span-first projection, independent of XML entity order.
    entities = sorted(ex['entities'], key=lambda e: (-(e['end']-e['start']), e['start'], e['end'], e['type']))
    for ent in entities:
        if not 0 <= ent['start'] < ent['end'] <= len(ex['text']):
            issues.append(dict(reason='invalid_offset', entity=ent, fatal=True))
            continue
        key = (ent['start'], ent['end'], ent['type'])
        if key in seen:
            issues.append(dict(reason='duplicate_annotation', entity=ent, action='deduplicate'))
            continue
        seen.add(key)
        positions = [i for i, (s, e) in enumerate(offsets) if s < ent['end'] and e > ent['start']]
        if not positions:
            issues.append(dict(reason='no_token_for_entity', entity=ent, fatal=True))
            continue
        projected = dict(start=offsets[positions[0]][0], end=offsets[positions[-1]][1], type=ent['type'])
        if projected['start'] != ent['start'] or projected['end'] != ent['end']:
            issues.append(dict(reason='boundary_inside_token', entity=ent, projected=projected,
                               token_offsets=[offsets[i] for i in positions], action='expand_to_token_boundaries'))
            if policy == 'strict':
                continue
        shared = [i for i in positions if i in owners]
        if shared:
            issues.append(dict(reason='overlapping_entities', entity=ent,
                               conflicting_entities=[owners[i] for i in shared], action='drop_conflicting_span'))
            continue
        for j, i in enumerate(positions):
            labels[i] = label2id[f"{'B' if j == 0 else 'I'}-{ent['type']}"]
            owners[i] = ent
    return dict(text=ex['text'], words=words, offsets=offsets, labels=labels), issues

In [5]:
def audit_splits(splits, label2id, report_path, policy="strict"):
    from collections import Counter
    import warnings
    if policy not in {"strict", "project"}:
        raise ValueError(f"Unknown annotation policy: {policy}")
    prepared, report, summary = {}, {}, {}
    for name, examples in splits.items():
        prepared[name], report[name] = [], []
        for ex in examples:
            row, issues = prepare_example(ex, label2id, policy=policy)
            prepared[name].append(row)
            if issues:
                report[name].append(dict(id=ex.get('id'), document=ex.get('document'),
                                         text=ex['text'], issues=issues))
        counts = Counter(issue['reason'] for row in report[name] for issue in row['issues'])
        summary[name] = dict(sentences=len(examples), affected_sentences=len(report[name]),
                             reasons=dict(counts))
        print(f"{name}: {len(report[name])}/{len(examples)} câu cần xử lý; {dict(counts)}")
    report['_policy'] = policy
    report['_summary'] = summary
    Path(report_path).write_text(json.dumps(report, ensure_ascii=False, indent=2))
    affected = sum(info['affected_sentences'] for info in summary.values())
    fatal = any(issue.get('fatal', False) for name in splits for row in report[name] for issue in row['issues'])
    if fatal or (policy == 'strict' and affected):
        raise ValueError(f'Annotation không biểu diễn chính xác bằng token BIO. Xem {report_path}. '
                         'Chế độ project chỉ xử lý ranh giới/trùng/xung đột; không bỏ qua offset sai hoặc câu rỗng.')
    if affected:
        warnings.warn(f'Đã project annotation trong {affected} câu; xem {report_path}. '
                      'F1 dùng nhãn đã điều chỉnh, không phải toàn bộ annotation gốc.')
    return prepared

In [6]:
def word_chunks(words, tokenizer, max_length):
    budget = max_length - tokenizer.num_special_tokens_to_add(pair=False)
    start, used = 0, 0
    for i, word in enumerate(words):
        size = len(tokenizer([word], is_split_into_words=True, add_special_tokens=False)['input_ids'])
        if size < 1 or size > budget:
            raise ValueError(f'Token không vừa cửa sổ Transformer: {word!r}, {size} pieces')
        if used + size > budget:
            yield start, i
            start, used = i, 0
        used += size
    if start < len(words):
        yield start, len(words)

In [7]:
TRAIN_DIR_DrugBank = Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Train/DrugBank")
TRAIN_DIR_MedLine = Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Train/MedLine")

TEST_ROOT = Path("/kaggle/input/datasets/tuantc2306/dataset/DDICorpus/Test")
TEST_DIR = TEST_ROOT / "Test for DrugNER task"


In [8]:
from pathlib import Path
from sklearn.model_selection import train_test_split


# 1. Chia danh sách file của từng nguồn
def split_xml_files(folder, dev_size=0.1, seed=42):
    xml_files = sorted(Path(folder).rglob("*.xml"))

    train_files, dev_files = train_test_split(
        xml_files,
        test_size=dev_size,
        random_state=seed,
        shuffle=True,
    )

    return train_files, dev_files
train_files_drug, dev_files_drug = split_xml_files(
    TRAIN_DIR_DrugBank, seed= SEED
)

train_files_med, dev_files_med = split_xml_files(
    TRAIN_DIR_MedLine, seed= SEED
)
test_files = sorted(TEST_DIR.rglob("*.xml"))

In [9]:
def read_xml_file(path):
    examples = []
    for sent in ET.parse(path).getroot().iter("sentence"):
        text = sent.attrib.get("text", "")
        if not text.strip():
            if sent.findall("entity"):
                raise ValueError(f"Annotation trong câu rỗng: {path}")
            continue
        entities = []
        for ent in sent.findall("entity"):
            kind = ent.attrib["type"]
            if kind not in ENTITY_TYPES:
                raise ValueError(f"Loại entity không hỗ trợ: {kind}")
            for span in ent.attrib["charOffset"].split(";"):
                start, last = map(int, span.split("-"))
                end = last + 1
                if not 0 <= start < end <= len(text):
                    raise ValueError(f"Offset XML không hợp lệ: {path}, {span}")
                entities.append(dict(start=start, end=end, type=kind, text=text[start:end]))
        examples.append(dict(id=sent.attrib.get("id", ""), document=str(path),
                             text=text, entities=entities))
    return examples

# 2. Đọc các câu từ danh sách file đã chia
def load_examples_from_files(xml_files):
    examples = []

    for xml_file in xml_files:
        examples.extend(read_xml_file(xml_file))

    return examples
train_examples_drug = load_examples_from_files(train_files_drug)
dev_examples_drug = load_examples_from_files(dev_files_drug)

train_examples_med = load_examples_from_files(train_files_med)
dev_examples_med = load_examples_from_files(dev_files_med)
# 3. Gộp hai nguồn
train_examples = train_examples_drug + train_examples_med
dev_examples = dev_examples_drug + dev_examples_med

# 4. Giữ nguyên tập test có sẵn
test_examples = load_examples_from_files(test_files)

In [10]:
# 5. Kiểm tra không trùng file giữa train và dev
train_files = train_files_drug + train_files_med
dev_files = dev_files_drug + dev_files_med

assert set(train_files).isdisjoint(dev_files)

for source, train_part, dev_part in [
    ("DrugBank", train_examples_drug, dev_examples_drug),
    ("MedLine", train_examples_med, dev_examples_med),
]:
    total = len(train_part) + len(dev_part)
    print(
        f"{source}: train={len(train_part)} câu | "
        f"dev={len(dev_part)} câu | "
        f"dev chiếm {len(dev_part) / total:.2%} số câu"
    )

print(f"\nTổng train: {len(train_examples)} câu")
print(f"Tổng dev:   {len(dev_examples)} câu")
print(f"Tổng test:  {len(test_examples)} câu")
manifest = dict(seed=SEED, unit="xml_document", dev_fraction=0.1,
                train=[str(p) for p in train_files], dev=[str(p) for p in dev_files],
                test=[str(p) for p in test_files])
(OUTPUT_DIR / "split_manifest.json").write_text(json.dumps(manifest, indent=2))

DrugBank: train=4928 câu | dev=676 câu | dev chiếm 12.06% số câu
MedLine: train=1124 câu | dev=177 câu | dev chiếm 13.60% số câu

Tổng train: 6052 câu
Tổng dev:   853 câu
Tổng test:  665 câu


78508

In [ ]:
class WordPieceSource(nn.Module):
    """Only BERT's input embedding table, never the Transformer encoder."""
    def __init__(self, model_name, device, trainable=False):
        super().__init__()
        from transformers import AutoModel, AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        
        pretrained = AutoModel.from_pretrained(model_name)
        self.embedding = pretrained.get_input_embeddings()
        del pretrained
        self.embedding.requires_grad_(trainable).to(device)
        self.dim = self.embedding.embedding_dim

    def forward(self, sentences):
        results = []
        for words in sentences:
            encoded = self.tokenizer(
                        words,
                        is_split_into_words=True,
                        add_special_tokens=False,
                        return_tensors='pt',
                        truncation=True,
                        max_length=258,            
                        return_attention_mask=False,
                        return_token_type_ids=False,
                        verbose=False,
                    )
            pieces = self.embedding(encoded['input_ids'].to(self.embedding.weight.device))[0]
            positions = [[] for _ in words]
            for j, word_id in enumerate(encoded.word_ids()):
                if word_id is not None:
                    positions[word_id].append(j)
            if not positions or any(not indices for indices in positions):
                raise ValueError('Câu rỗng hoặc tokenizer làm mất token')
            results.append(torch.stack([pieces[indices].mean(0) for indices in positions]))
        return results

In [12]:
class FastTextSource(nn.Module):
    """Train-vocabulary table initialized from FastText; OOV uses frozen subwords."""
    def __init__(self, model_path, vocab, trainable=False):
        super().__init__()
        import fasttext
        import numpy as np
        self.fasttext = fasttext.load_model(str(model_path))
        self.dim = self.fasttext.get_dimension()
        self.vocab = list(vocab)
        if not self.vocab or len(set(self.vocab)) != len(self.vocab):
            raise ValueError('FastText vocabulary phải khác rỗng và không trùng')
        self.word2id = {word: i for i, word in enumerate(self.vocab)}
        initial = torch.from_numpy(np.stack([
            self.fasttext.get_word_vector(word) for word in self.vocab
        ])).float()
        self.embedding = nn.Embedding.from_pretrained(initial, freeze=not trainable)

    def forward(self, sentences):
        results = []
        device = self.embedding.weight.device
        for words in sentences:
            rows = []
            for word in words:
                word = word.lower()
                index = self.word2id.get(word)
                if index is None:
                    # Do not grow the vocabulary using dev/test. Preserve FastText OOV support.
                    vector = torch.as_tensor(self.fasttext.get_word_vector(word),
                                             dtype=self.embedding.weight.dtype, device=device)
                else:
                    vector = self.embedding(torch.tensor(index, device=device))
                rows.append(vector)
            results.append(torch.stack(rows))
        return results

In [13]:
class FlairSource(nn.Module):
    def __init__(self, names, device, trainable=False):
        super().__init__()
        import flair
        from flair.embeddings import FlairEmbeddings, StackedEmbeddings
        flair.device = device
        self.trainable = trainable
        self.model = StackedEmbeddings([
            FlairEmbeddings(name, fine_tune=trainable) for name in names
        ])
        self.model.requires_grad_(trainable).to(device)
        self.dim = self.model.embedding_length
        self.train(self.training)

    def train(self, mode=True):
        super().train(mode)
        if not self.trainable:
            self.model.eval()
        return self

    def forward(self, words_batch):
        import flair
        from flair.data import Sentence
        flair.device = next(self.model.parameters()).device
        # Fresh objects each forward: no stale embeddings or autograd graphs.
        sentences = [Sentence(words) for words in words_batch]
        track_grad = self.trainable and torch.is_grad_enabled()
        # Flair can enable gradients internally; explicitly disable fine_tune for evaluation.
        embeddings = list(self.model.embeddings)
        previous = [embedding.fine_tune for embedding in embeddings]
        try:
            for embedding in embeddings:
                embedding.fine_tune = track_grad
            with torch.set_grad_enabled(track_grad):
                self.model.embed(sentences)
                results = [torch.stack([token.embedding for token in sentence])
                           for sentence in sentences]
        finally:
            for embedding, fine_tune in zip(embeddings, previous):
                embedding.fine_tune = fine_tune
            for sentence in sentences:
                sentence.clear_embeddings()
        return results

In [14]:
class TokenDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        return dict(words=row['words'], tags=torch.tensor(row['labels'], dtype=torch.long))


In [15]:
def collate_words(items):
    words = [item['words'] for item in items]
    lengths = torch.tensor([len(item['tags']) for item in items])
    if (lengths <= 0).any() or any(len(w) != len(item['tags']) for w, item in zip(words, items)):
        raise ValueError('Câu rỗng hoặc số token không khớp nhãn')
    tags = pad_sequence([item['tags'] for item in items], batch_first=True)
    mask = torch.arange(tags.size(1))[None, :] < lengths[:, None]
    return dict(words=words, tags=tags, mask=mask, lengths=lengths)


def move_batch(batch, device):
    return dict(words=batch['words'], tags=batch['tags'].to(device),
                mask=batch['mask'].to(device), lengths=batch['lengths'])


In [16]:
class SourceAttention(nn.Module):
    def __init__(self, source_dims, fusion_dim):
        super().__init__()
        if not source_dims:
            raise ValueError('Cần ít nhất một nguồn embedding')
        self.names = list(source_dims)
        self.projections = nn.ModuleDict({name: nn.Sequential(nn.Linear(dim, fusion_dim),
                                         nn.LayerNorm(fusion_dim)) for name, dim in source_dims.items()})
        self.source_bias = nn.Parameter(torch.zeros(len(self.names), fusion_dim))
        self.score = nn.Sequential(nn.Linear(fusion_dim, fusion_dim), nn.Tanh(), nn.Linear(fusion_dim, 1, bias=False))

    def forward(self, features, mask):
        projected = torch.stack([self.projections[name](features[name]) for name in self.names], dim=2)
        weights = self.score(projected + self.source_bias).squeeze(-1).softmax(dim=2)
        weights = weights.masked_fill(~mask.unsqueeze(-1), 0)
        return (weights.unsqueeze(-1) * projected).sum(2), weights

In [19]:
class FusionBiLSTMCRF(nn.Module):
    def __init__(self, sources, source_dims, num_tags, fusion_dim=200, hidden_dim=100, dropout=0.3):
        super().__init__()
        self.sources = nn.ModuleDict(sources)
        self.fusion = SourceAttention(source_dims, fusion_dim)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(fusion_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(2 * hidden_dim, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, batch, return_attention=False):
        features = {}
        for name, source in self.sources.items():
            vectors = source(batch['words'])
            features[name] = pad_sequence(vectors, batch_first=True)
        fused, weights = self.fusion(features, batch['mask'])
        packed = pack_padded_sequence(self.dropout(fused), batch['lengths'].cpu(), batch_first=True, enforce_sorted=False)
        output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(output, batch_first=True, total_length=batch['mask'].size(1))
        emissions = self.classifier(self.dropout(output))
        return (emissions, weights) if return_attention else emissions

    def loss(self, batch):
        return -self.crf(self(batch), batch['tags'], mask=batch['mask'], reduction='mean')

    def decode(self, batch):
        return self.crf.decode(self(batch), mask=batch['mask'])


In [37]:
def train_epoch(model, loader, optimizer, device, epoch=None, total_epochs=None):
    from tqdm.auto import tqdm
    model.train()
    total_loss = 0.0
    
    # Tiêu đề thanh chạy batch (ví dụ: Epoch 01/40)
    desc = f"Epoch {epoch:02d}/{total_epochs}" if epoch and total_epochs else "Training"
    
    # Bọc loader vào tqdm: leave=False giúp thanh chạy tự thu gọn/ẩn khi xong 1 epoch để đỡ rối màn hình
    batch_pbar = tqdm(loader, desc=desc, leave=False, unit="batch")
    
    for raw in batch_pbar:
        batch = move_batch(raw, device)
        optimizer.zero_grad(set_to_none=True)
        loss = model.loss(batch)
        if not torch.isfinite(loss):
            raise RuntimeError('Nonfinite loss')
        loss.backward()
        optimizer.step()
        
        batch_loss = loss.item()
        total_loss += batch_loss
        
        # Cập nhật loss của batch hiện tại theo thời gian thực
        batch_pbar.set_postfix({"batch_loss": f"{batch_loss:.4f}"})
        
    return total_loss / len(loader) if len(loader) > 0 else 0.0

In [35]:
@torch.no_grad()
def evaluate_model(model, loader, labels, device):
    from seqeval.metrics import precision_score, recall_score, f1_score
    from seqeval.scheme import IOB2
    model.eval()
    gold, pred = [], []
    total_loss = 0.0

    for raw in loader:
        batch = move_batch(raw, device)
        
        # Tính validation loss
        batch_loss = model.loss(batch)
        total_loss += batch_loss.item()

        for row, path in enumerate(model.decode(batch)):
            gold.append([labels[i] for i in batch['tags'][row, :len(path)].tolist()])
            pred.append([labels[i] for i in path])
            
    kwargs = dict(mode='strict', scheme=IOB2, zero_division=0)
    avg_loss = total_loss / len(loader) if len(loader) > 0 else 0.0

    return dict(
        loss=float(avg_loss),
        precision=float(precision_score(gold, pred, **kwargs)),
        recall=float(recall_score(gold, pred, **kwargs)),
        f1=float(f1_score(gold, pred, **kwargs))
    )

In [23]:

prepared = audit_splits(dict(train=train_examples, dev=dev_examples, test=test_examples),
                        label2id, OUTPUT_DIR / 'annotation_report.json', policy=ANNOTATION_POLICY)


train: 66/6052 câu cần xử lý; {'boundary_inside_token': 50, 'overlapping_entities': 24, 'duplicate_annotation': 9}
dev: 11/853 câu cần xử lý; {'boundary_inside_token': 9, 'overlapping_entities': 5}
test: 2/665 câu cần xử lý; {'boundary_inside_token': 2, 'overlapping_entities': 2}


/tmp/ipykernel_58/3856598739.py:28: UserWarning: Đã project annotation trong 79 câu; xem /kaggle/working/multi_embedding_fusion/biomedbert-flair-fasttext-online-biomedbert0-flair0-fasttext0-project/annotation_report.json. F1 dùng nhãn đã điều chỉnh, không phải toàn bộ annotation gốc.
  warnings.warn(f'Đã project annotation trong {affected} câu; xem {report_path}. '


In [24]:
# Chỉ khởi tạo và tải các nguồn được chọn; không tải model của nguồn đã tắt.
from huggingface_hub import hf_hub_download
sources = {}
for name in ENABLED_SOURCES:
    if name == 'biomedbert':
        sources[name] = WordPieceSource(MODEL_NAME, DEVICE, TRAINABLE_SOURCES[name])
    elif name == 'flair':
        sources[name] = FlairSource(FLAIR_NAMES, DEVICE, TRAINABLE_SOURCES[name])
    elif name == 'fasttext':
        if not FASTTEXT_PATH.is_file():
            if not DOWNLOAD_FASTTEXT:
                raise FileNotFoundError(FASTTEXT_PATH)
            FASTTEXT_PATH = Path(hf_hub_download('facebook/fasttext-en-vectors', 'model.bin',
                                               local_dir=str(FASTTEXT_PATH.parent)))
        vocab = sorted({w.lower() for row in prepared['train'] for w in row['words']})
        sources[name] = FastTextSource(FASTTEXT_PATH, vocab, TRAINABLE_SOURCES[name])
SOURCE_DIMS = {name: source.dim for name, source in sources.items()}
FEATURE_CONFIG = dict(annotation_policy=ANNOTATION_POLICY, sources=ENABLED_SOURCES, transformer=MODEL_NAME, flair=FLAIR_NAMES,
                      fasttext=str(FASTTEXT_PATH), token_pattern=TOKEN_PATTERN,
                      biomedbert_representation='input_embeddings',
                      trainable_sources={name: TRAINABLE_SOURCES[name] for name in ENABLED_SOURCES},
                      fasttext_representation='train_vocab_table_with_frozen_oov',
                      fasttext_vocab=sources['fasttext'].vocab if 'fasttext' in sources else [], pooling='mean_wordpiece_lookup',
                      fasttext_lowercase=True, alignment='regex_exact_boundaries_v1')
MODEL_CONFIG = dict(source_dims=SOURCE_DIMS, num_tags=len(LABELS), fusion_dim=200, hidden_dim=64, dropout=0.5)
# print('Dimensions:', SOURCE_DIMS)


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

2026-09-25 06:31:39,747 https://flair.informatik.hu-berlin.de/resources/embeddings/flair/pubmed-forward.pt not found in cache, downloading to /tmp/tmpyclp0vkz



  0%|          | 0.00/69.4M [00:00<?, ?B/s]
  0%|          | 8.00k/69.4M [00:00<16:57, 71.6kB/s]
  0%|          | 32.0k/69.4M [00:00<07:57, 152kB/s] 
  0%|          | 94.0k/69.4M [00:00<03:41, 328kB/s]
  0%|          | 180k/69.4M [00:00<02:27, 492kB/s] 
  1%|          | 383k/69.4M [00:00<01:16, 946kB/s]
  1%|          | 782k/69.4M [00:00<00:40, 1.79MB/s]
  2%|▏         | 1.54M/69.4M [00:00<00:20, 3.45MB/s]
  4%|▍         | 3.11M/69.4M [00:00<00:10, 6.75MB/s]
  8%|▊         | 5.65M/69.4M [00:01<00:05, 12.0MB/s]
 10%|█         | 7.26M/69.4M [00:01<00:04, 13.4MB/s]
 13%|█▎        | 8.83M/69.4M [00:01<00:04, 14.2MB/s]
 15%|█▌        | 10.4M/69.4M [00:01<00:04, 15.0MB/s]
 17%|█▋        | 12.0M/69.4M [00:01<00:03, 15.5MB/s]
 20%|██        | 13.9M/69.4M [00:01<00:03, 16.8MB/s]
 23%|██▎       | 15.8M/69.4M [00:01<00:03, 17.7MB/s]
 26%|██▌       | 17.7M/69.4M [00:01<00:02, 18.3MB/s]
 28%|██▊       | 19.6M/69.4M [00:01<00:02, 18.8MB/s]
 31%|███▏      | 21.7M/69.4M [00:01<00:02, 19.8MB/s]
 34%|█

2026-09-25 06:31:44,936 copying /tmp/tmpyclp0vkz to cache at /root/.flair/embeddings/pubmed-forward.pt
2026-09-25 06:31:44,994 removing temp file /tmp/tmpyclp0vkz


2026-09-25 06:31:46,226 https://flair.informatik.hu-berlin.de/resources/embeddings/flair/pubmed-backward.pt not found in cache, downloading to /tmp/tmpvpxq757i



  0%|          | 0.00/69.4M [00:00<?, ?B/s]
  0%|          | 8.00k/69.4M [00:00<17:06, 70.9kB/s]
  0%|          | 32.0k/69.4M [00:00<08:00, 151kB/s] 
  0%|          | 94.0k/69.4M [00:00<03:43, 326kB/s]
  0%|          | 196k/69.4M [00:00<02:13, 543kB/s] 
  1%|          | 422k/69.4M [00:00<01:09, 1.05MB/s]
  1%|          | 860k/69.4M [00:00<00:36, 1.97MB/s]
  2%|▏         | 1.71M/69.4M [00:00<00:18, 3.82MB/s]
  5%|▍         | 3.43M/69.4M [00:00<00:09, 7.41MB/s]
  8%|▊         | 5.84M/69.4M [00:01<00:05, 11.7MB/s]
 12%|█▏        | 8.43M/69.4M [00:01<00:04, 15.1MB/s]
 15%|█▌        | 10.7M/69.4M [00:01<00:03, 17.5MB/s]
 18%|█▊        | 12.4M/69.4M [00:01<00:03, 17.2MB/s]
 20%|██        | 14.1M/69.4M [00:01<00:03, 17.0MB/s]
 24%|██▎       | 16.4M/69.4M [00:01<00:02, 19.0MB/s]
 26%|██▌       | 18.2M/69.4M [00:01<00:02, 18.8MB/s]
 30%|██▉       | 20.6M/69.4M [00:01<00:02, 20.7MB/s]
 33%|███▎      | 23.1M/69.4M [00:01<00:02, 22.1MB/s]
 37%|███▋      | 25.5M/69.4M [00:02<00:01, 23.1MB/s]
 40%|

2026-09-25 06:31:51,170 copying /tmp/tmpvpxq757i to cache at /root/.flair/embeddings/pubmed-backward.pt


2026-09-25 06:31:51,219 removing temp file /tmp/tmpvpxq757i


model.bin:   0%|          | 0.00/7.24G [00:00<?, ?B/s]

In [25]:
def make_loader(rows, shuffle=False):
    dataset = TokenDataset(rows)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
                      collate_fn=collate_words, generator=torch.Generator().manual_seed(SEED))
train_loader = make_loader(prepared['train'], shuffle=True)
dev_loader = make_loader(prepared['dev'])
model = FusionBiLSTMCRF(sources=sources, **MODEL_CONFIG).to(DEVICE)
# Lower learning rates for pretrained embeddings; only trainable parameters enter AdamW.
parameter_groups = []
head = [p for name, p in model.named_parameters()
        if not name.startswith('sources.') and p.requires_grad]
parameter_groups.append(dict(params=head, lr=LR))
for name, source in model.sources.items():
    params = [p for p in source.parameters() if p.requires_grad]
    if params:
        parameter_groups.append(dict(params=params, lr=EMBEDDING_LRS[name]))
    print(name, '| trainable parameters:', sum(p.numel() for p in params))
optimizer = torch.optim.AdamW(parameter_groups, weight_decay=1e-5)
batch = move_batch(next(iter(train_loader)), DEVICE)
model.eval()
with torch.no_grad():
    emissions, attention = model(batch, return_attention=True)
print('Emissions:', emissions.shape, '| Attention [B,T,S]:', attention.shape)
print('Sources:', model.fusion.names)
assert torch.allclose(attention.sum(-1), batch['mask'].float(), atol=1e-6)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


biomedbert | trainable parameters: 0
flair | trainable parameters: 0
fasttext | trainable parameters: 0
Emissions: torch.Size([32, 58, 9]) | Attention [B,T,S]: torch.Size([32, 58, 3])
Sources: ['biomedbert', 'flair', 'fasttext']
Trainable parameters: 1213052


In [27]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("attention")

In [31]:
import wandb

WANDB_PROJECT = "ddi-ner-embedAttention_BiLSTM_CRF"
WANDB_RUN_NAME = f"{MODEL_NAME.split('/')[-1]}-seed{SEED}"


def wandb_config():
    return {
        "architecture": "BiLSTM + CRF",
        "train_batch_size": 32,
        "eval_batch_size": 32,
        "epochs": 40,
        "LR": 1e-3,
        "seed": SEED,
        "num_labels": len(LABELS),
        "labels": LABELS,
        "train_sentences": len(train_examples),
        "valid_sentences": len(dev_examples),
        "test_sentences": len(test_examples),
    }


def ensure_wandb_run():
    if wandb.run is None:
        run = wandb.init(
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config=wandb_config(),
        )
        wandb.watch(model, log="gradients", log_freq=100)
        return run
    return wandb.run

wandb.login(key=secret_value_0)
wandb_run = ensure_wandb_run()


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [39]:
import pandas as pd

In [38]:
from tqdm.auto import tqdm

best_f1, stale_epochs, history = -1.0, 0, []
BEST_PATH = OUTPUT_DIR / 'best.pt'

# Thanh tiến trình tổng quản lý các Epoch
epoch_pbar = tqdm(range(1, EPOCHS + 1), desc="Training Progress", unit="epoch")

for epoch in epoch_pbar:
    # 1. Chạy train epoch
    train_loss = train_epoch(model, train_loader, optimizer, DEVICE, epoch=epoch, total_epochs=EPOCHS)
    
    # 2. Đánh giá tập train và tập dev
    train_eval = evaluate_model(model, train_loader, LABELS, DEVICE)
    dev_eval = evaluate_model(model, dev_loader, LABELS, DEVICE)
    
    # Gom dữ liệu lại
    row = dict(
        epoch=epoch,
        **{f'train_{k}': v for k, v in train_eval.items()},
        **{f'dev_{k}': v for k, v in dev_eval.items()}
    )
    history.append(row)
    
    # Log vào Weights & Biases
    wandb.log(row, step=epoch)
    
    # In kết quả sau mỗi epoch dưới dạng format rõ ràng
    print(f"\n Epoch {epoch:02d}/{EPOCHS} Summary:")
    print(f"  - Train -> Loss: {train_loss:.4f} | Prec: {train_eval['precision']:.4f} | Rec: {train_eval['recall']:.4f} | F1: {train_eval['f1']:.4f}")
    print(f"  - Dev   -> Loss: {dev_eval['loss']:.4f} | Prec: {dev_eval['precision']:.4f} | Rec: {dev_eval['recall']:.4f} | F1: {dev_eval['f1']:.4f}")

    # Cập nhật thông tin ngắn gọn lên chính thanh tiến trình
    epoch_pbar.set_postfix({"Dev F1": f"{dev_eval['f1']:.4f}", "Dev Loss": f"{dev_eval['loss']:.4f}"})

    # Lưu best model & check early stopping
    if dev_eval['f1'] > best_f1:
        best_f1, stale_epochs = dev_eval['f1'], 0
        wandb.run.summary["best_dev_f1"] = best_f1
        wandb.run.summary["best_epoch"] = epoch
        
        torch.save(dict(model_state_dict=model.state_dict(), model_config=MODEL_CONFIG,
                        feature_config=FEATURE_CONFIG, labels=LABELS, seed=SEED,
                        epoch=epoch, dev_f1=best_f1, split_manifest=manifest), BEST_PATH)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"\n[!] Kích hoạt Early stopping tại epoch {epoch} (Patience = {PATIENCE})")
            break

print(f'\n Best dev F1: {best_f1:.4f}')


Training Progress:   0%|          | 0/40 [00:00<?, ?epoch/s]

Epoch 01/40:   0%|          | 0/190 [00:00<?, ?batch/s]


 Epoch 01/40 Summary:
  - Train -> Loss: 1.3484 | Prec: 0.9465 | Rec: 0.9411 | F1: 0.9438
  - Dev   -> Loss: 1.4725 | Prec: 0.9244 | Rec: 0.8835 | F1: 0.9035


Epoch 02/40:   0%|          | 0/190 [00:00<?, ?batch/s]


 Epoch 02/40 Summary:
  - Train -> Loss: 1.0276 | Prec: 0.9413 | Rec: 0.9641 | F1: 0.9526
  - Dev   -> Loss: 1.4635 | Prec: 0.9023 | Rec: 0.9136 | F1: 0.9079


...

Epoch 08/40:   0%|          | 0/190 [00:00<?, ?batch/s]


 Epoch 08/40 Summary:
  - Train -> Loss: 0.4523 | Prec: 0.9774 | Rec: 0.9836 | F1: 0.9805
  - Dev   -> Loss: 1.5446 | Prec: 0.9347 | Rec: 0.9113 | F1: 0.9229


Epoch 09/40:   0%|          | 0/190 [00:00<?, ?batch/s]


 Epoch 09/40 Summary:
  - Train -> Loss: 0.4271 | Prec: 0.9799 | Rec: 0.9849 | F1: 0.9824
  - Dev   -> Loss: 1.5755 | Prec: 0.9288 | Rec: 0.9119 | F1: 0.9203

[!] Kích hoạt Early stopping tại epoch 9 (Patience = 5)

 Best dev F1: 0.9262


NameError: name 'pd' is not defined

In [41]:
history_df = pd.DataFrame(history)

In [43]:
wandb.finish()
history_df

dev_f1,▁▂▅█▇▅▄▇▆
dev_loss,▃▃█▁▁▄▇▅▅
dev_precision,▆▁▆█▅█▃█▆
dev_recall,▁▇▅▇█▄▆▆▆
epoch,▁▂▃▄▅▅▆▇█
train_f1,▁▃▄▆▆▇▇██
train_loss,█▆▅▃▃▂▂▁▁
train_precision,▂▁▃▅▅▇▆██
train_recall,▁▅▄▇▇▆███
best_dev_f1,0.92617
best_epoch,4


,epoch,train_loss,train_precision,train_recall,train_f1,dev_loss,dev_precision,dev_recall,dev_f1
0,1,0.872923,0.946489,0.941104,0.943789,1.472517,0.924450,0.883457,0.903488
1,2,0.703035,0.941296,0.964093,0.952558,1.463515,0.902302,0.913587,0.907910
2,3,0.648874,0.954920,0.962556,0.958723,1.715229,0.925044,0.905060,0.914943
3,4,0.446629,0.963841,0.975550,0.969660,1.365340,0.936121,0.916430,0.926171
4,5,0.462775,0.962666,0.979394,0.970958,1.364520,0.923121,0.921546,0.922333
5,6,0.352364,0.976246,0.973243,0.974742,1.503287,0.934397,0.898806,0.916256
6,7,0.336160,0.967986,0.983392,0.975628,1.662465,0.913538,0.913019,0.913278
7,8,0.268063,0.977386,0.983623,0.980494,1.544620,0.934694,0.911313,0.922855
8,9,0.247155,0.979881,0.984853,0.982361,1.575466,0.928778,0.911882,0.920252


In [44]:
print(BEST_PATH)

/kaggle/working/multi_embedding_fusion/biomedbert-flair-fasttext-online-biomedbert0-flair0-fasttext0-project/best.pt


In [46]:
test_loader = make_loader(prepared['test'])
test_eval = evaluate_model(model, test_loader, LABELS, DEVICE)

In [47]:
print(test_eval)

{'loss': 2.066690524419149, 'precision': 0.7688022284122563, 'recall': 0.8046647230320699, 'f1': 0.7863247863247863}


In [49]:

# 1. Load checkpoint tốt nhất
checkpoint = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

# 2. Đánh giá trên tập test
test_eval = evaluate_model(model, test_loader, LABELS, DEVICE)

# 3. Hiển thị kết quả
print(f"Test Loss:      {test_eval['loss']:.4f}")
print(f"Test Precision: {test_eval['precision']:.4f}")
print(f"Test Recall:    {test_eval['recall']:.4f}")
print(f"Test F1-score:  {test_eval['f1']:.4f}")

Test Loss:      1.8197
Test Precision: 0.7289
Test Recall:    0.7682
Test F1-score:  0.7480


## Error analysis — Drug NER
Chạy lần lượt các cell dưới đây sau khi đã chuẩn bị dữ liệu và khởi tạo `model`; **không cần train lại**. Nếu kernel đã reset, chạy lại phần cấu hình, định nghĩa hàm/lớp, đọc dữ liệu và khởi tạo model trước khi chạy phần này; bỏ qua vòng lặp training.

- Mặc định phân tích `test` bằng checkpoint `best.pt` được chọn theo dev. Đổi `EA_SPLIT = "dev"` nếu dùng kết quả để cải tiến mô hình; không chọn model/hyperparameter dựa trên test.
- Đánh giá **exact entity match, strict IOB2** giống `evaluate_model`: đúng cả ranh giới và loại thực thể.
- Với `ANNOTATION_POLICY="project"`, gold là nhãn đã điều chỉnh. XML có thực thể gián đoạn đang được reader tách thành các span riêng; đây không phải đánh giá trực tiếp toàn bộ annotation XML gốc.
- Các cell mới chỉ suy luận và xuất báo cáo, không cập nhật trọng số bằng optimizer.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
from seqeval.metrics import classification_report, f1_score
from seqeval.scheme import IOB2, Tokens

EA_SPLIT = "test"  # Đổi thành "dev" khi phân tích để cải tiến mô hình.
EA_CHECKPOINT = Path(BEST_PATH)  # Có thể đổi sang best.pt đã gắn từ Kaggle Dataset.
EA_DIR = Path(OUTPUT_DIR) / "error_analysis" / EA_SPLIT

if EA_SPLIT not in {"dev", "test"}:
    raise ValueError("EA_SPLIT phải là dev hoặc test")
if not EA_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {EA_CHECKPOINT}")

ea_checkpoint = torch.load(EA_CHECKPOINT, map_location="cpu", weights_only=True)
if ea_checkpoint["labels"] != LABELS:
    raise ValueError("Thứ tự LABELS không khớp checkpoint")
if ea_checkpoint["feature_config"] != FEATURE_CONFIG:
    raise ValueError("FEATURE_CONFIG không khớp checkpoint; kiểm tra nguồn embedding/vocab/policy")
if ea_checkpoint["split_manifest"] != manifest:
    raise ValueError("Danh sách file split không khớp checkpoint")
model.load_state_dict(ea_checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

ea_rows = prepared[EA_SPLIT]
ea_examples = {"dev": dev_examples, "test": test_examples}[EA_SPLIT]
assert len(ea_rows) == len(ea_examples) > 0
assert all(r["text"] == e["text"] for r, e in zip(ea_rows, ea_examples))
EA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Split: {EA_SPLIT} | Câu: {len(ea_rows)} | Checkpoint epoch: {ea_checkpoint['epoch']}")
print("Thư mục báo cáo:", EA_DIR)

### 1. Thu thập dự đoán theo đúng thứ tự câu
Dùng `model.decode()` để lấy đường đi CRF, không lấy argmax của emissions. Loader không shuffle và loại padding theo chiều dài thật. Chỉ cell này chạy suy luận; các cell sau tái sử dụng dự đoán.

In [ ]:
@torch.no_grad()
def ea_collect_predictions(model, rows):
    model.eval()
    gold, pred = [], []
    loader = make_loader(rows, shuffle=False)
    for raw in tqdm(loader, desc="Error analysis: inference"):
        paths = model.decode(move_batch(raw, DEVICE))
        assert len(paths) == len(raw["words"])
        for i, path in enumerate(paths):
            n = int(raw["lengths"][i])
            assert len(path) == n
            gold.append([LABELS[j] for j in raw["tags"][i, :n].tolist()])
            pred.append([LABELS[j] for j in path])
    assert len(gold) == len(pred) == len(rows)
    assert all(len(g) == len(p) == len(r["words"])
               for r, g, p in zip(rows, gold, pred))
    return gold, pred


ea_gold, ea_pred = ea_collect_predictions(model, ea_rows)
print(classification_report(ea_gold, ea_pred, mode="strict", scheme=IOB2,
                            digits=4, zero_division=0))

### 2. Chuẩn bị thực thể, metadata và hàm phân loại lỗi
Mỗi thực thể có dạng `(token_start, token_end_exclusive, type)`. Các vị trí ký tự xuất ra cũng dùng khoảng `[start, end)`.

- `missed_entity`: gold không có prediction hợp lệ chồng lấn; `spurious_entity`: prediction không có gold chồng lấn.
- `wrong_type`: đúng ranh giới, sai loại; `wrong_boundary`: chồng lấn cùng loại nhưng khác ranh giới; `wrong_boundary_and_type`: khác cả hai.
- `split_merge`: một span chồng lấn nhiều span phía đối diện. Đây là gợi ý để đọc thủ công, không phải phép ghép cặp tối ưu.

Một lỗi sai loại/ranh giới thường tạo ra **một FN và một FP**. Bảng đếm theo FN/FP, không coi tổng số dòng là số sự cố độc lập. Một số chuỗi `I-*` sai quy tắc không tạo thành entity trong strict IOB2; xem thêm bảng lỗi BIO ở cuối.

In [ ]:
def ea_spans(tags):
    return {(e.start, e.end, e.tag) for e in Tokens(tags, scheme=IOB2).entities}


def ea_span_info(span, row):
    start, end, kind = span
    char_start = row["offsets"][start][0]
    char_end = row["offsets"][end - 1][1]
    return dict(text=row["text"][char_start:char_end], type=kind,
                start=char_start, end=char_end, token_length=end - start)


def ea_source(example):
    identity = (example.get("document", "") + " " + example.get("id", "")).lower()
    if "drugbank" in identity:
        return "DrugBank"
    if "medline" in identity:
        return "MedLine"
    return "unknown"


def ea_error_category(span, others, side):
    overlaps = sorted(o for o in others if span[0] < o[1] and o[0] < span[1])
    if not overlaps:
        return ("missed_entity" if side == "FN" else "spurious_entity"), overlaps
    if any(span[:2] == o[:2] for o in overlaps):
        return "wrong_type", overlaps  # Exact matches đã được loại trước đó.
    if len(overlaps) > 1:
        return "split_merge", overlaps
    category = "wrong_boundary" if span[2] == overlaps[0][2] else "wrong_boundary_and_type"
    return category, overlaps


ea_gold_spans = [ea_spans(tags) for tags in ea_gold]
ea_pred_spans = [ea_spans(tags) for tags in ea_pred]
ea_metadata = []
for i, (row, example) in enumerate(zip(ea_rows, ea_examples)):
    checked_row, issues = prepare_example(example, label2id, policy=ANNOTATION_POLICY)
    assert checked_row["labels"] == row["labels"], "Prepared labels không khớp annotation policy"
    ea_metadata.append(dict(
        sentence_index=i, sentence_id=example.get("id", ""),
        document=example.get("document", ""), source=ea_source(example),
        annotation_affected=bool(issues),
        annotation_reasons=sorted({issue["reason"] for issue in issues}),
    ))

### 3. Precision / recall / F1 theo loại và nguồn
`support` là số gold entity, `predicted` là số entity dự đoán. Các nhóm không có entity dùng quy ước điểm bằng 0. So sánh tỷ lệ và support, không chỉ số lỗi tuyệt đối. Các dòng `annotation_affected` giúp kiểm tra riêng những câu bị ảnh hưởng bởi bước xử lý annotation.

In [ ]:
def ea_metrics(indices, entity_type=None):
    tp = fp = fn = 0
    for i in indices:
        gold = {s for s in ea_gold_spans[i] if entity_type is None or s[2] == entity_type}
        pred = {s for s in ea_pred_spans[i] if entity_type is None or s[2] == entity_type}
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return dict(sentences=len(indices), support=tp + fn, predicted=tp + fp,
                tp=tp, fp=fp, fn=fn, precision=precision, recall=recall, f1=f1)


ea_all_indices = list(range(len(ea_rows)))
ea_metric_records = [dict(group="overall", value="all", **ea_metrics(ea_all_indices))]
for kind in ENTITY_TYPES:
    ea_metric_records.append(dict(group="entity_type", value=kind,
                                  **ea_metrics(ea_all_indices, kind)))
for field in ["source", "annotation_affected"]:
    for value in sorted({m[field] for m in ea_metadata}):
        indices = [i for i, m in enumerate(ea_metadata) if m[field] == value]
        ea_metric_records.append(dict(group=field, value=str(value), **ea_metrics(indices)))
ea_metrics_df = pd.DataFrame(ea_metric_records)
assert abs(ea_metrics_df.iloc[0]["f1"] - f1_score(
    ea_gold, ea_pred, mode="strict", scheme=IOB2, zero_division=0)) < 1e-12

display(ea_metrics_df.round(4))

### 4. Bảng lỗi thực thể và lỗi BIO
`overlapping_other` chứa prediction khi dòng là FN, hoặc gold khi dòng là FP. Giữ toàn bộ span chồng lấn để đọc được cả trường hợp tách/gộp. `manual_category` và `notes` để bạn ghi nhận nguyên nhân sau khi đọc ngữ cảnh.

In [ ]:
ea_error_records, ea_bio_records = [], []
for i, row in enumerate(ea_rows):
    gold, pred = ea_gold_spans[i], ea_pred_spans[i]
    for side, unmatched, others in [("FN", gold - pred, pred), ("FP", pred - gold, gold)]:
        for span in sorted(unmatched):
            category, overlaps = ea_error_category(span, others, side)
            ea_error_records.append({
                **ea_metadata[i], "error": side, "category": category,
                **ea_span_info(span, row),
                "overlapping_other": [ea_span_info(o, row) for o in overlaps],
                "sentence": row["text"], "manual_category": "", "notes": "",
            })
    for j, tag in enumerate(ea_pred[i]):
        previous = ea_pred[i][j - 1] if j else "O"
        if tag.startswith("I-") and previous not in {"B-" + tag[2:], "I-" + tag[2:]}:
            ea_bio_records.append(dict(
                sentence_index=i, sentence_id=ea_metadata[i]["sentence_id"],
                token_index=j, token=row["words"][j], previous_pred=previous,
                pred=tag, gold=ea_gold[i][j], sentence=row["text"],
            ))

ea_errors_df = pd.DataFrame(ea_error_records, columns=[
    "sentence_index", "sentence_id", "document", "source", "annotation_affected",
    "annotation_reasons", "error", "category", "text", "type", "start", "end",
    "token_length", "overlapping_other", "sentence", "manual_category", "notes",
])
ea_bio_errors_df = pd.DataFrame(ea_bio_records, columns=[
    "sentence_index", "sentence_id", "token_index", "token", "previous_pred", "pred", "gold", "sentence",
])
ea_error_summary = (ea_errors_df.groupby(["error", "category", "type"])
                    .size().rename("count").reset_index())
assert int((ea_errors_df["error"] == "FN").sum()) == ea_metrics_df.iloc[0]["fn"]
assert int((ea_errors_df["error"] == "FP").sum()) == ea_metrics_df.iloc[0]["fp"]
display(ea_error_summary)
display(ea_errors_df.head(20))
print("Số vị trí vi phạm IOB2:", len(ea_bio_errors_df))
display(ea_bio_errors_df.head(10))

### 5. Đọc câu cụ thể và các cụm từ thường bị sai
Gọi `ea_inspect_sentence(index)` với `sentence_index` trong bảng lỗi. Cột `wrong` chỉ là lỗi ở mức token để hỗ trợ đọc; không dùng làm thay thế F1 ở mức entity.

Ví dụ lọc: `ea_errors_df.query("category == 'wrong_type'")` hoặc `ea_errors_df.query("source == 'MedLine' and error == 'FN'")`.

In [ ]:
def ea_inspect_sentence(index):
    if not 0 <= index < len(ea_rows):
        raise IndexError(index)
    print(ea_metadata[index])
    print(ea_rows[index]["text"])
    tokens = pd.DataFrame({
        "token": ea_rows[index]["words"], "offset": ea_rows[index]["offsets"],
        "gold": ea_gold[index], "pred": ea_pred[index],
    })
    tokens["wrong"] = tokens["gold"] != tokens["pred"]
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(tokens)
        display(ea_errors_df[ea_errors_df["sentence_index"] == index])


if not ea_errors_df.empty:
    ea_inspect_sentence(int(ea_errors_df.iloc[0]["sentence_index"]))
else:
    print("Không có FP/FN ở mức entity; vẫn kiểm tra bảng lỗi BIO nếu có.")

ea_frequent_errors = (ea_errors_df.groupby(["error", "category", "type", "text"])
                      .size().rename("count").sort_values(ascending=False).reset_index())
display(ea_frequent_errors.head(20))

### 6. Lưu báo cáo
CSV dùng UTF-8 BOM để dễ mở bằng Excel. `predictions.json` giữ toàn bộ câu, token, offset, gold và prediction để phân tích lại mà không cần suy luận lần nữa. Chạy lại sẽ ghi đè báo cáo của cùng `EA_SPLIT` trong thư mục `error_analysis`.

In [ ]:
ea_export_errors = ea_errors_df.copy()
for column in ["annotation_reasons", "overlapping_other"]:
    ea_export_errors[column] = ea_export_errors[column].map(
        lambda value: json.dumps(value, ensure_ascii=False))

for filename, frame in {
    "metrics.csv": ea_metrics_df,
    "entity_errors.csv": ea_export_errors,
    "error_summary.csv": ea_error_summary,
    "bio_errors.csv": ea_bio_errors_df,
    "frequent_errors.csv": ea_frequent_errors,
}.items():
    frame.to_csv(EA_DIR / filename, index=False, encoding="utf-8-sig")

ea_saved = dict(
    split=EA_SPLIT, checkpoint=str(EA_CHECKPOINT), epoch=ea_checkpoint["epoch"],
    checkpoint_dev_f1=float(ea_checkpoint["dev_f1"]), labels=LABELS,
    annotation_policy=ANNOTATION_POLICY, evaluation="strict IOB2 exact entity match",
    sentences=[{
        **meta, "text": row["text"], "words": row["words"], "offsets": row["offsets"],
        "gold": gold, "pred": pred,
    } for meta, row, gold, pred in zip(ea_metadata, ea_rows, ea_gold, ea_pred)],
)
(EA_DIR / "predictions.json").write_text(
    json.dumps(ea_saved, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu báo cáo:")
for report_path in sorted(EA_DIR.iterdir()):
    if report_path.is_file():
        print(report_path)